<a href="https://colab.research.google.com/github/juliamarcella/Women-in-the-Olympics-Game/blob/main/Gender%20gap%20Olympics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = kagglehub.dataset_download("piterfm/olympic-games-medals-19862018")

# Load the latest version
print("Path to dataset files:", file_path)

In [ ]:
import os

print(file_path)
print(os.listdir(file_path))

In [ ]:
import pandas as pd
import numpy as np

df_athletes= pd.read_csv(os.path.join(file_path, "olympic_athletes.csv"))
df_hosts = pd.read_csv(os.path.join(file_path, "olympic_hosts.csv"))
df_medals = pd.read_csv(os.path.join(file_path, "olympic_medals.csv"))
df_results = pd.read_csv(os.path.join(file_path, "olympic_results.csv"))
print(df_athletes.info())
print(df_hosts.info())
print(df_medals.info())
print(df_results.info())


## df_athletes

In [ ]:
df_athletes.sample(5)

In [ ]:
df_athletes["athlete_year_birth"] = df_athletes["athlete_year_birth"].astype('Int64')
df_athletes.sample(5)

In [ ]:
df_athletes["athlete_medals"].value_counts()

In [ ]:
import re

def parse_medals(s):
    if pd.isna(s):
        return pd.Series({'Gold': 0, 'Silver': 0, 'Bronze': 0})

    s = re.sub(r'\s+', '', s)  # remove whitespace

    # Find patterns like "5G", "3S", "2B"
    gold = sum(int(n) for n in re.findall(r'(\d+)G', s))
    silver = sum(int(n) for n in re.findall(r'(\d+)S', s))
    bronze = sum(int(n) for n in re.findall(r'(\d+)B', s))

    return pd.Series({'Gold': gold, 'Silver': silver, 'Bronze': bronze})

df_athletes[['Gold', 'Silver', 'Bronze']] = df_athletes['athlete_medals'].apply(parse_medals)
df_athletes['Total_medals'] = df_athletes['Gold'] + df_athletes['Silver'] + df_athletes['Bronze']


In [ ]:
df_athletes = df_athletes.drop(columns="athlete_medals")

In [ ]:
df_athletes.sample(10)

In [ ]:
df_athletes.isnull().sum()

In [ ]:
df_athletes['first_game'] = df_athletes['first_game'].fillna('Unknown')
df_athletes['games_participations'] = df_athletes['games_participations'].fillna(0)
df_athletes['bio'] = df_athletes['bio'].fillna('No biography available')
df_athletes["athlete_year_birth"] = df_athletes["athlete_year_birth"].fillna(0)

In [ ]:
df_athletes.isnull().sum()

In [ ]:
df_athletes[df_athletes.duplicated(keep=False)]

## *df_hosts*

In [ ]:
df_hosts.sample(5)

In [ ]:
df_hosts.info(0)

In [ ]:
df_hosts["game_end_date"] = pd.to_datetime(df_hosts["game_end_date"])
df_hosts["game_start_date"] = pd.to_datetime(df_hosts["game_start_date"])
df_hosts["game_year"] = df_hosts["game_year"].astype("Int64")

text_cols = ['game_slug', 'game_location', 'game_name', 'game_season']

for col in text_cols:
    df_hosts[col] = df_hosts[col].astype('string').str.strip()


df_hosts.info()

## Df_medals

In [ ]:
df_medals.info(0)

In [ ]:
df_medals.sample(5)

In [ ]:
text_cols = [
    'discipline_title', 'slug_game', 'event_title', 'event_gender',
    'medal_type', 'participant_type', 'athlete_full_name',
    'country_name', 'country_code', 'country_3_letter_code'
]

for col in text_cols:
    if col in df_medals.columns:
        df_medals[col] = (
            df_medals[col]
            .astype("string")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

In [ ]:
df_medals.isnull().sum()

In [ ]:
df_medals["event_gender"].value_counts()

In [ ]:
df_medals['is_team_event'] = df_medals['participant_type'].eq('gameteam')
df_medals['medal_type'] = df_medals['medal_type'].str.upper()

In [ ]:
df_medals[(df_medals["athlete_full_name"].isna()) &
          (df_medals["participant_type"] == "gameteam")]


In [ ]:
df_medals.info()

In [ ]:
df_medals.describe()


## Df_results

In [ ]:
df_results.info()

In [ ]:
df_results.sample(5)

In [ ]:
df_results["participant_type"].value_counts()

In [ ]:
text_cols = [
    'discipline_title', 'event_title', 'slug_game', 'participant_type',
    'athletes', 'rank_equal', 'rank_position',
    'country_name', 'country_code', 'country_3_letter_code',
    'athlete_url', 'athlete_full_name', 'value_unit', 'value_type'
]

for col in text_cols:
    if col in df_medals.columns:
        df_results[col] = (
            df_results[col]
            .astype("string")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

In [ ]:
df_results['medal_type'] = df_results['medal_type'].str.upper()

In [ ]:
m = df_results['medal_type'].fillna('')

df_results['Gold']   = (m == 'GOLD').astype(int)
df_results['Silver'] = (m == 'SILVER').astype(int)
df_results['Bronze'] = (m == 'BRONZE').astype(int)

In [ ]:
df_results['is_team_event'] = df_results['participant_type'].eq('gameteam')

In [ ]:
df_results.info()
df_results.head(10)

In [ ]:
def extract_gender(title):
    title = title.lower()

    if "women" in title or "ladies" in title:
        return "women"
    if "men" in title:
        return "men"
    if "mixed" in title:
        return "mixed"
    # Luge doubles, sailing open, croquet open, shooting open
    return "open"

df_results['event_gender'] = df_results['event_title'].apply(extract_gender)

In [ ]:
df_results['athlete_gender'] = df_results['event_gender']

In [ ]:
df_results.loc[df_results['is_team_event'], 'athlete_gender'] = 'team'

In [ ]:
df_results.sample(2)

In [ ]:
df_results.info(0)

In [ ]:
df_athletes.info()

## . Mrge Datasets

In [ ]:
# Step 1: Merge df_results with df_hosts
# Merge on game identifiers, using left join to keep all results entries.
# Suffixes are added to distinguish columns from df_results and df_hosts.
df_final = pd.merge(df_results, df_hosts, left_on='slug_game', right_on='game_slug', how='left', suffixes=('_results', '_hosts'))
# Drop the redundant 'game_slug' column after the merge
df_final.drop(columns=['game_slug'], inplace=True)

# Step 2: Merge df_final (results + hosts) with df_athletes
# Merge on 'athlete_url' to bring in athlete-specific details like birth year, total medals, etc.
# Suffixes are used for potential conflicts in columns like 'athlete_full_name' or 'Gold/Silver/Bronze' total medals.
df_final = pd.merge(df_final, df_athletes, on='athlete_url', how='left', suffixes=('_event_details', '_athlete_details'))

# Step 3: Merge df_final (results + hosts + athletes) with specific columns from df_medals
# This step aims to enrich medal-winning rows with 'participant_title' from df_medals,
# which can be more explicit for team events. We select specific columns from df_medals
# to avoid excessive redundancy and rename 'participant_title' for clarity.
df_medals_to_add = df_medals[['slug_game', 'discipline_title', 'event_title', 'medal_type', 'athlete_url', 'country_name', 'participant_title']].copy()
df_medals_to_add.rename(columns={'participant_title': 'medal_event_participant_title'}, inplace=True)

# Merge this temporary df_medals_to_add into df_final.
# The merge keys are chosen to uniquely identify a medal entry across both dataframes.
# The columns from df_results (e.g., 'slug_game', 'discipline_title') were not suffixed in previous merges for these specific names,
# so we refer to them directly.
df_final = pd.merge(df_final,
                    df_medals_to_add,
                    left_on=['slug_game', 'discipline_title', 'event_title', 'athlete_url', 'country_name', 'medal_type'],
                    right_on=['slug_game', 'discipline_title', 'event_title', 'athlete_url', 'country_name', 'medal_type'],
                    how='left',
                    suffixes=('_final', '_medals_details'))

# Drop the redundant columns that were used for merging from df_medals_to_add
df_final.drop(columns=[
    'slug_game_medals_details', 'discipline_title_medals_details', 'event_title_medals_details',
    'athlete_url_medals_details', 'country_name_medals_details', 'medal_type_medals_details'
], inplace=True, errors='ignore')

print(df_final.info())



In [ ]:
cols_keep = [
    "discipline_title",
    "event_title",
    "game_year",
    "game_season",
    "game_location",
    "game_name",
    "participant_type",
    "is_team_event",
    "medal_type",
    "Gold_event_details",
    "Silver_event_details",
    "Bronze_event_details",
    "country_name",
    "country_3_letter_code",
    "athlete_full_name_event_details",
    "athlete_gender",
    "event_gender"
]
df_clean = df_final[cols_keep].copy()


In [ ]:
df_clean.sample(5)

In [ ]:
df_clean["athlete_gender"].value_counts()

In [ ]:
df_clean.info(0)

### Analize of Df_Clean

In [ ]:
df_clean.groupby("event_gender")[["Gold_event_details", "Silver_event_details", "Bronze_event_details"]].sum()

In [ ]:
df_clean.groupby(["game_name", "athlete_gender"])[["Gold_event_details", "Silver_event_details", "Bronze_event_details"]].sum()

In [ ]:
df_clean.groupby(["game_year", "athlete_gender"])["Gold_event_details"].sum().unstack()

In [ ]:
first_women = (
    df_clean[df_clean["event_gender"] == "women"]
    .groupby("discipline_title")["game_year"]
    .min()
    .rename("first_women_year")
)


In [ ]:
first_men = (
    df_clean[df_clean["event_gender"] == "men"]
    .groupby("discipline_title")["game_year"]
    .min()
    .rename("first_men_year")
)

In [ ]:
gender_start_years = (
    pd.concat([first_men, first_women], axis=1)
    .sort_values("first_women_year")
)

In [ ]:
gender_start_years["years_difference"] = (
    gender_start_years["first_women_year"] - gender_start_years["first_men_year"]
)

gender_start_years

In [ ]:
disciplines_with_gaps = gender_start_years[gender_start_years['years_difference'].notna()].copy()
disciplines_with_gaps = disciplines_with_gaps.sort_values(by='years_difference', ascending=False)
print("Disciplines with calculated gender participation gaps (sorted by difference):\n")
print(disciplines_with_gaps.head())

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 8))
plt.barh(disciplines_with_gaps['first_men_year'].head(15).index, disciplines_with_gaps['years_difference'].head(15), color='skyblue')
plt.xlabel('Years Difference')
plt.ylabel('Discipline')
plt.title('Top 15 Olympic Disciplines with Largest Gender Participation Gaps')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
mixed_gender_disciplines = df_clean[df_clean['event_gender'] == 'mixed']['discipline_title'].unique()
sports_without_women = gender_start_years[gender_start_years['first_women_year'].isna()]

# Filter out disciplines that have mixed events
sports_without_women_filtered = sports_without_women[~sports_without_women.index.isin(mixed_gender_disciplines)]

print("\nSports where women have not yet participated (gender-specific events, excluding mixed events):")
for discipline in sports_without_women_filtered.index:
    print(discipline)

In [ ]:
mask = df_clean["event_title"].str.contains(r"\b(women|woman|ladies|female|girls)\b", case=True, regex=True, na=False)
df_filtered = df_clean[mask]

In [ ]:
df_filtered["athlete_gender"].value_counts()

In [ ]:
df_clean.info(0)

In [ ]:
df_filtered["event_gender"].value_counts()

In [ ]:
def classify_gender(row):
    title = row["event_title"].lower()

    if any(w in title for w in ["women", "woman", "ladies", "female", "girls"]):
        return "women"
    if any(m in title for m in ["men", "man", "male", "boys"]):
        return "men"

    # Everything else becomes mixed
    return "mixed"

df_clean["athlete_gender"] = df_clean.apply(classify_gender, axis=1)

In [ ]:
df_clean["athlete_gender"].value_counts()

In [ ]:
df_clean.to_csv("df_clean.csv", index=False)

In [ ]:
df_clean.sample(10)

In [ ]:
df_clean.info()

In [ ]:
medals_by_year_gender = df_clean.groupby(['game_year', 'athlete_gender'])[['Gold_event_details', 'Silver_event_details', 'Bronze_event_details']].sum().reset_index()
print(medals_by_year_gender.head())

In [ ]:
medals_pivot = medals_by_year_gender.pivot_table(
    index='game_year',
    columns='athlete_gender',
    values=['Gold_event_details', 'Silver_event_details', 'Bronze_event_details'],
    aggfunc='sum'
)

medals_pivot = medals_pivot.fillna(0)

medals_pivot['men_Total_Medals'] = medals_pivot['Gold_event_details']['men'] + medals_pivot['Silver_event_details']['men'] + medals_pivot['Bronze_event_details']['men']
medals_pivot['women_Total_Medals'] = medals_pivot['Gold_event_details']['women'] + medals_pivot['Silver_event_details']['women'] + medals_pivot['Bronze_event_details']['women']

print(medals_pivot.head())

In [ ]:
correlation_men_women = medals_pivot['men_Total_Medals'].corr(medals_pivot['women_Total_Medals'])
print(f"Correlation between men's and women's total medals: {correlation_men_women}")

In [ ]:
participation_by_year_gender = df_clean.groupby(['game_year', 'athlete_gender'])['athlete_full_name_event_details'].nunique().reset_index()
participation_by_year_gender.rename(columns={'athlete_full_name_event_details': 'num_participants'}, inplace=True)
print(participation_by_year_gender.head())

In [ ]:
participation_by_gender_pivot = participation_by_year_gender.pivot_table(
    index='game_year',
    columns='athlete_gender',
    values='num_participants',
    aggfunc='sum'
).rename(columns={'men': 'men_num_participants', 'women': 'women_num_participants'})

print(participation_by_gender_pivot.head())

**Reasoning**:
Now that the participation data is pivoted, I will merge it with the `medals_pivot` DataFrame on the `game_year` index and fill any resulting NaN values with 0, as per the instructions.



In [ ]:
medals_pivot = medals_by_year_gender.pivot_table(
    index='game_year',
    columns='athlete_gender',
    values=['Gold_event_details', 'Silver_event_details', 'Bronze_event_details'],
    aggfunc='sum'
)
# Flatten the MultiIndex columns to a single level
medals_pivot.columns = [f'{col[0]}_{col[1]}' for col in medals_pivot.columns]
medals_pivot = medals_pivot.fillna(0)

# Re-add the Total_Medals columns using the flattened names
medals_pivot['men_Total_Medals'] = medals_pivot['Gold_event_details_men'] + medals_pivot['Silver_event_details_men'] + medals_pivot['Bronze_event_details_men']
medals_pivot['women_Total_Medals'] = medals_pivot['Gold_event_details_women'] + medals_pivot['Silver_event_details_women'] + medals_pivot['Bronze_event_details_women']

# Re-create participation_by_gender_pivot, ensuring it only contains desired columns and is flat.
participation_by_gender_pivot = participation_by_year_gender.pivot_table(
    index='game_year',
    columns='athlete_gender',
    values='num_participants',
    aggfunc='sum'
)
# Rename only the 'men' and 'women' columns for clarity and to match naming convention
participation_by_gender_pivot.rename(columns={'men': 'men_num_participants', 'women': 'women_num_participants'}, inplace=True)
# Select only the men's and women's participation columns, dropping 'mixed' if it exists
participation_by_gender_pivot = participation_by_gender_pivot[['men_num_participants', 'women_num_participants']]
participation_by_gender_pivot = participation_by_gender_pivot.fillna(0) # Fill NaN for years without specific gender participation.

# Now perform the merge with flattened columns
medals_pivot = pd.merge(medals_pivot, participation_by_gender_pivot, left_index=True, right_index=True, how='left')
medals_pivot = medals_pivot.fillna(0)
print(medals_pivot.head())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 10), sharex=True)

# Plot Medals
axes[0].plot(medals_pivot.index, medals_pivot['men_Total_Medals'], label="Men's Medals", marker='o', markersize=4, color='blue')
axes[0].plot(medals_pivot.index, medals_pivot['women_Total_Medals'], label="Women's Medals", marker='o', markersize=4, color='red')
axes[0].set_ylabel('Total Medals')
axes[0].set_title('Olympic Medals and Participation Trends by Gender Over the Years')
axes[0].legend()
axes[0].grid(True)

# Plot Participation
axes[1].plot(medals_pivot.index, medals_pivot['men_num_participants'], label="Men's Participants", marker='o', markersize=4, color='darkblue')
axes[1].plot(medals_pivot.index, medals_pivot['women_num_participants'], label="Women's Participants", marker='o', markersize=4, color='darkred')
axes[1].set_xlabel('Game Year')
axes[1].set_ylabel('Number of Participants')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
participation_by_discipline_gender = df_clean.groupby(['discipline_title', 'athlete_gender'])['athlete_full_name_event_details'].nunique().reset_index()
participation_by_discipline_gender.rename(columns={'athlete_full_name_event_details': 'num_participants'}, inplace=True)
print(participation_by_discipline_gender.head())

In [ ]:
participation_by_discipline_pivot = participation_by_discipline_gender.pivot_table(
    index='discipline_title',
    columns='athlete_gender',
    values='num_participants',
    aggfunc='sum'
).rename(columns={'men': 'men_num_participants', 'women': 'women_num_participants'})

# Keep only men's and women's participation, filling NaN with 0
participation_by_discipline_pivot = participation_by_discipline_pivot[['men_num_participants', 'women_num_participants']].fillna(0)
print(participation_by_discipline_pivot.head())

In [ ]:
participation_by_discipline_pivot['total_participants'] = participation_by_discipline_pivot['men_num_participants'] + participation_by_discipline_pivot['women_num_participants']

participation_by_discipline_sorted = participation_by_discipline_pivot.sort_values(by='total_participants', ascending=False)

top_n = 20
participation_to_plot = participation_by_discipline_sorted.head(top_n)

fig, ax = plt.subplots(figsize=(14, 10))

bar_width = 0.35
index = range(len(participation_to_plot))

bar1 = ax.barh(index, participation_to_plot['men_num_participants'], bar_width, label="Men's Participants", color='skyblue')
bar2 = ax.barh([i + bar_width for i in index], participation_to_plot['women_num_participants'], bar_width, label="Women's Participants", color='lightcoral')

ax.set_xlabel('Number of Participants')
ax.set_ylabel('Discipline')
ax.set_title(f'Top {top_n} Olympic Disciplines by Gender Participation')
ax.set_yticks([i + bar_width / 2 for i in index])
ax.set_yticklabels(participation_to_plot.index)
ax.legend()
ax.invert_yaxis() # Display top discipline at the top

plt.tight_layout()
plt.show()